## 0. Setup

In [ ]:
!pip install -q duckdb shap

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

ROOT = Path('/content/drive/MyDrive/olist-delivery-delay')   # survives session resets
RAW = Path('/content/data/raw')                              # re-downloadable, no need to keep on Drive
PROC = ROOT / "data" / "processed"                           # parquet, model tables, predictions, json
FIG = ROOT / "reports" / "figures"                           # charts only
MODELS = ROOT / "models"
TAB = PROC / "tableau"                                       # files for Tableau
for p in (RAW, PROC, FIG, MODELS, TAB):
    p.mkdir(parents=True, exist_ok=True)

Mounted at /content/drive


In [ ]:
from google.colab import files
files.upload()
!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

Saving kaggle.json to kaggle.json


In [ ]:
!kaggle datasets download -d olistbr/brazilian-ecommerce -p /content/data/raw --unzip
!ls /content/data/raw

Dataset URL: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce
License(s): CC-BY-NC-SA-4.0
100% 42.6M/42.6M [00:00<00:00, 59.1MB/s]

olist_customers_dataset.csv	  olist_orders_dataset.csv
olist_geolocation_dataset.csv	  olist_products_dataset.csv
olist_order_items_dataset.csv	  olist_sellers_dataset.csv
olist_order_payments_dataset.csv  product_category_name_translation.csv
olist_order_reviews_dataset.csv


In [ ]:
# Shared settings
TARGET = "is_late"

BASE_NUM = ["n_items", "n_sellers", "n_products", "total_price", "total_freight", "freight_ratio",
            "total_weight_g", "total_volume_cm3", "payment_value", "max_installments",
            "purchase_month", "purchase_dow", "purchase_hour", "approval_delay_h", "promised_days"]
BASE_CAT = ["primary_category_en", "main_payment_type"]
GEO_NUM = ["distance_km", "same_state"]
GEO_CAT = ["customer_state", "seller_state"]
HIST_NUM = ["seller_prior_orders", "seller_prior_late_rate"]

NUM = BASE_NUM + GEO_NUM + HIST_NUM
CAT = BASE_CAT + GEO_CAT
FEATURES = NUM + CAT

LGBM_PARAMS = dict(n_estimators=2000, learning_rate=0.03, num_leaves=31, min_child_samples=50,
                   subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0,
                   random_state=42, verbose=-1)


def load_model_table():
    df = pd.read_parquet(PROC / "model_table.parquet")
    df = df.sort_values("purchase_ts").reset_index(drop=True)
    for c in CAT:
        df[c] = df[c].astype("category")
    return df


def time_split(df, train=0.70, val=0.15):
    """Chronological split. df must be sorted by purchase_ts."""
    n = len(df)
    i1, i2 = int(n * train), int(n * (train + val))
    return df.iloc[:i1], df.iloc[i1:i2], df.iloc[i2:]

## 1. SQL layer: one row per delivered order

In [ ]:
import duckdb

FILES = {
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "product_category_name_translation": "product_category_name_translation.csv",
}

con = duckdb.connect(database=":memory:")
for name, f in FILES.items():
    con.execute(f"CREATE OR REPLACE TABLE {name} AS SELECT * FROM read_csv_auto('{RAW / f}')")
    print(name, con.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0])

sql_zip_geo = """
CREATE OR REPLACE TABLE zip_geo AS
SELECT geolocation_zip_code_prefix AS zip,
       AVG(geolocation_lat) AS lat,
       AVG(geolocation_lng) AS lng
FROM geolocation
GROUP BY 1;
"""
con.execute(sql_zip_geo)

sql_order_level = """
CREATE OR REPLACE TABLE order_level AS
WITH items AS (
    SELECT oi.order_id,
           COUNT(*)                                   AS n_items,
           COUNT(DISTINCT oi.seller_id)               AS n_sellers,
           COUNT(DISTINCT oi.product_id)              AS n_products,
           SUM(oi.price)                              AS total_price,
           SUM(oi.freight_value)                      AS total_freight,
           SUM(p.product_weight_g)                    AS total_weight_g,
           SUM(p.product_length_cm * p.product_height_cm * p.product_width_cm) AS total_volume_cm3,
           ARG_MIN(oi.seller_id, oi.order_item_id)    AS primary_seller_id,
           ARG_MIN(p.product_category_name, oi.order_item_id) AS primary_category
    FROM order_items oi
    LEFT JOIN products p USING (product_id)
    GROUP BY oi.order_id
),
pay AS (
    SELECT order_id,
           SUM(payment_value)                 AS payment_value,
           MAX(payment_installments)          AS max_installments,
           ARG_MAX(payment_type, payment_value) AS main_payment_type
    FROM order_payments
    GROUP BY order_id
),
rev AS (
    SELECT order_id, ARG_MAX(review_score, review_creation_date) AS review_score
    FROM order_reviews
    GROUP BY order_id
)
SELECT
    o.order_id,
    o.customer_id,
    c.customer_unique_id,
    c.customer_state,
    i.primary_seller_id,
    s.seller_state,
    ct.product_category_name_english           AS primary_category_en,
    i.n_items, i.n_sellers, i.n_products,
    i.total_price, i.total_freight, i.total_weight_g, i.total_volume_cm3,
    pay.payment_value, pay.max_installments, pay.main_payment_type,
    cg.lat AS c_lat, cg.lng AS c_lng,
    sg.lat AS s_lat, sg.lng AS s_lng,
    CAST(o.order_purchase_timestamp AS TIMESTAMP)       AS purchase_ts,
    CAST(o.order_approved_at AS TIMESTAMP)              AS approved_ts,
    CAST(o.order_delivered_carrier_date AS TIMESTAMP)   AS carrier_ts,
    CAST(o.order_delivered_customer_date AS TIMESTAMP)  AS delivered_ts,
    CAST(o.order_estimated_delivery_date AS TIMESTAMP)  AS estimated_ts,
    rev.review_score,
    DATE_DIFF('day', CAST(o.order_estimated_delivery_date AS DATE),
                     CAST(o.order_delivered_customer_date AS DATE)) AS delay_days,
    CASE WHEN CAST(o.order_delivered_customer_date AS DATE)
              > CAST(o.order_estimated_delivery_date AS DATE) THEN 1 ELSE 0 END AS is_late
FROM orders o
JOIN items i          ON i.order_id = o.order_id
JOIN customers c      ON c.customer_id = o.customer_id
LEFT JOIN sellers s   ON s.seller_id = i.primary_seller_id
LEFT JOIN product_category_name_translation ct ON ct.product_category_name = i.primary_category
LEFT JOIN pay         ON pay.order_id = o.order_id
LEFT JOIN rev         ON rev.order_id = o.order_id
LEFT JOIN zip_geo cg  ON cg.zip = c.customer_zip_code_prefix
LEFT JOIN zip_geo sg  ON sg.zip = s.seller_zip_code_prefix
WHERE o.order_status = 'delivered'
  AND o.order_approved_at IS NOT NULL
  AND o.order_delivered_customer_date IS NOT NULL;
"""
con.execute(sql_order_level)

n, late = con.execute("SELECT COUNT(*), AVG(is_late) FROM order_level").fetchone()
print(f"order_level: {n:,} rows, late rate {late:.2%}")
con.execute(f"COPY order_level TO '{PROC / 'order_level.parquet'}' (FORMAT PARQUET)")

orders 99441
order_items 112650
order_payments 103886
order_reviews 99224
geolocation 1000163
products 32951
sellers 3095
customers 99441
product_category_name_translation 71
order_level: 96,456 rows, late rate 6.77%


## 2. Feature engineering

In [ ]:
df = pd.read_parquet(PROC / "order_level.parquet")


def haversine(lat1, lon1, lat2, lon2):
    """Great-circle distance in km between two points given in decimal degrees."""
    R = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi, dlmb = p2 - p1, np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dlmb / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))


# Geography
df["distance_km"] = haversine(df["c_lat"], df["c_lng"], df["s_lat"], df["s_lng"])
df["same_state"] = (df.customer_state == df.seller_state).astype(int)

# Order features known at approval
df["freight_ratio"] = df.total_freight / df.total_price.replace(0, np.nan)
df["purchase_month"] = df.purchase_ts.dt.month
df["purchase_dow"] = df.purchase_ts.dt.dayofweek
df["purchase_hour"] = df.purchase_ts.dt.hour
df["approval_delay_h"] = (df.approved_ts - df.purchase_ts).dt.total_seconds() / 3600
# Your definition (measured from purchase). Both purchase and estimated dates are known at approval, so no leakage.
df["promised_days"] = (df.estimated_ts - df.purchase_ts).dt.total_seconds() / 86400

# Time-aware seller history: only deliveries COMPLETED before this order's approval
hist = df[["primary_seller_id", "delivered_ts", "is_late"]].sort_values("delivered_ts").copy()
grp = hist.groupby("primary_seller_id")
hist["seller_prior_orders"] = grp.cumcount() + 1
hist["seller_prior_late_rate"] = grp["is_late"].cumsum() / hist["seller_prior_orders"]
hist = hist.rename(columns={"delivered_ts": "hist_ts"}).drop(columns="is_late")

df = df.sort_values("approved_ts")
df = pd.merge_asof(df, hist, left_on="approved_ts", right_on="hist_ts",
                   by="primary_seller_id", direction="backward", allow_exact_matches=False)
df["seller_prior_orders"] = df["seller_prior_orders"].fillna(0)
df = df.drop(columns="hist_ts").sort_values("purchase_ts").reset_index(drop=True)

df.to_parquet(PROC / "model_table.parquet", index=False)
print(df.shape)
print(df[["distance_km", "promised_days", "seller_prior_late_rate"]].describe().T)

(96456, 39)
                          count        mean         std       min         25%  \
distance_km             95978.0  600.749491  593.694042  0.000000  185.908397   
promised_days           96456.0   23.734715    8.760201  2.008009   18.328941   
seller_prior_late_rate  91295.0    0.044533    0.065104  0.000000    0.000000   

                               50%         75%          max  
distance_km             433.878565  798.612588  8677.911622  
promised_days            23.230289   28.406340   155.135463  
seller_prior_late_rate    0.031250    0.061404     1.000000  


## 3. SQL analysis (Q1 to Q3)

In [ ]:
try:
    con.execute("SELECT 1 FROM order_level LIMIT 1")
except Exception:
    con = duckdb.connect(database=":memory:")
    con.execute(f"CREATE TABLE order_level AS SELECT * FROM read_parquet('{PROC / 'order_level.parquet'}')")

sql_q1 = """
WITH m AS (
    SELECT DATE_TRUNC('month', purchase_ts) AS month,
           COUNT(*) AS orders, AVG(is_late) AS late_rate
    FROM order_level
    WHERE purchase_ts >= '2017-01-01'
    GROUP BY 1
)
SELECT month, orders,
       ROUND(100 * late_rate, 2) AS late_pct,
       ROUND(100 * AVG(late_rate) OVER (ORDER BY month ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 2) AS late_pct_3m
FROM m ORDER BY month;
"""

sql_q2 = """
SELECT primary_seller_id, seller_state, COUNT(*) AS orders,
       ROUND(100 * AVG(is_late), 1) AS late_pct,
       RANK() OVER (ORDER BY AVG(is_late) DESC) AS rnk
FROM order_level
GROUP BY 1, 2
HAVING COUNT(*) >= 30
ORDER BY rnk LIMIT 20;
"""

sql_q3 = """
SELECT CASE WHEN delay_days <= 0 THEN '1 on time or early'
            WHEN delay_days <= 3 THEN '2 late 1-3 days'
            WHEN delay_days <= 7 THEN '3 late 4-7 days'
            ELSE '4 late 8+ days' END AS bucket,
       COUNT(*) AS orders, ROUND(AVG(review_score), 2) AS avg_review
FROM order_level
WHERE review_score IS NOT NULL
GROUP BY 1 ORDER BY 1;
"""

for title, q in [("Q1: monthly late rate with 3-month rolling average", sql_q1),
                 ("Q2: worst sellers (min 30 orders)", sql_q2),
                 ("Q3: review score by delay bucket", sql_q3)]:
    print(title)
    display(con.execute(q).fetchdf())

Q1: monthly late rate with 3-month rolling average


,month,orders,late_pct,late_pct_3m
0,2017-01-01,748,2.94,2.94
1,2017-02-01,1641,2.99,2.96
2,2017-03-01,2546,4.56,3.49
3,2017-04-01,2303,6.56,4.70
4,2017-05-01,3545,2.99,4.70
5,2017-06-01,3135,3.03,4.19
6,2017-07-01,3872,2.79,2.94
7,2017-08-01,4193,2.91,2.91
8,2017-09-01,4150,4.39,3.36
9,2017-10-01,4478,4.18,3.82


Q2: worst sellers (min 30 orders)


,primary_seller_id,seller_state,orders,late_pct,rnk
0,ad781527c93d00d89a11eecd9dcad7c1,SP,35,34.3,1
1,ede0c03645598cdfc63ca8237acbe73d,SP,43,32.6,2
2,54965bbe3e4f07ae045b90b0b8541f52,PR,72,30.6,3
3,2a1348e9addc1af5aaa619b1a3679d6b,MG,48,27.1,4
4,835f0f7810c76831d6c7d24c7a646d4d,SP,41,26.8,5
5,beadbee30901a7f61d031b6b686095ad,SP,63,23.8,6
6,a49928bcdf77c55c6d6e05e09a9b4ca5,SP,94,22.3,7
7,ef990a83bbea832f36ebe81376335aa8,SC,42,21.4,8
8,712e6ed8aa4aa1fa65dab41fed5737e4,SC,77,20.8,9
9,3fd1e727ba94cfe122d165e176ce7967,SP,34,20.6,10


Q3: review score by delay bucket


,bucket,orders,avg_review
0,1 on time or early,89429,4.29
1,2 late 1-3 days,1852,3.29
2,3 late 4-7 days,1748,2.10
3,4 late 8+ days,2781,1.70


## 4. Exploratory analysis

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

df = pd.read_parquet(PROC / "model_table.parquet")
print(f"orders: {len(df):,} | late rate: {df.is_late.mean():.2%}")

# Monthly late rate
m = df[df.purchase_ts >= "2017-01-01"].set_index("purchase_ts").resample("MS")["is_late"].mean() * 100
plt.figure(figsize=(10, 4))
m.plot(color="#c0504d")
plt.title("Late delivery rate by purchase month")
plt.ylabel("Late rate (%)")
plt.xlabel("Purchase Month")
plt.tight_layout(); plt.savefig(FIG / "eda_monthly_late.png", dpi=200); plt.close()

# Late rate by customer state (minimum 100 orders)
s = df.groupby("customer_state").is_late.agg(["mean", "size"])
s = s[s["size"] >= 100].sort_values("mean", ascending=False)
s["mean"] = s["mean"] * 100
plt.figure(figsize=(10, 4))
s["mean"].plot.bar(color="#c0504d")
plt.title("Late rate by customer state"); plt.ylabel("% late")
plt.tight_layout(); plt.savefig(FIG / "eda_state_late.png", dpi=200); plt.close()

# Late rate by distance bucket
df["dist_bucket"] = pd.cut(df.distance_km, [0, 100, 300, 600, 1000, 2000, 5000])
plt.figure(figsize=(7, 4))
(df.groupby("dist_bucket", observed=True).is_late.mean() * 100).plot.bar()
plt.title("Late rate by seller-customer distance (km)"); plt.ylabel("% late")
plt.tight_layout(); plt.savefig(FIG / "eda_distance_late.png", dpi=200); plt.close()

# Promise length: late vs on time
plt.figure(figsize=(7, 4))
sns.boxplot(data=df, x="is_late", y="promised_days", showfliers=False)
plt.title("Promised delivery window (days) by outcome")
plt.tight_layout(); plt.savefig(FIG / "eda_promise_window.png", dpi=200); plt.close()

# Review score by late flag
plt.figure(figsize=(7, 4))
sns.barplot(data=df.dropna(subset=["review_score"]), x="is_late", y="review_score", errorbar="ci")
plt.title("Average review score: on time (0) vs late (1)")
plt.tight_layout(); plt.savefig(FIG / "eda_review_by_late.png", dpi=200); plt.close()

# Missing values and duplicates
print(df[["distance_km", "total_weight_g", "primary_category_en", "review_score"]].isna().mean().round(3))
print("duplicate order_id:", df.order_id.duplicated().sum())

orders: 96,456 | late rate: 6.77%
distance_km            0.005
total_weight_g         0.000
primary_category_en    0.014
review_score           0.007
dtype: float64
duplicate order_id: 0


## 5. Models: logistic regression, LightGBM, XGBoost, cost-based threshold

In [ ]:
import joblib
import lightgbm as lgb
import xgboost as xgb
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = load_model_table()
tr, va, te = time_split(df)
print(f"train {len(tr):,} | val {len(va):,} | test {len(te):,}")
print("late rate train/val/test:", [round(x[TARGET].mean(), 4) for x in (tr, va, te)])
print("date ranges:", [(x.purchase_ts.min().date(), x.purchase_ts.max().date()) for x in (tr, va, te)])

X_tr, y_tr = tr[FEATURES], tr[TARGET].values
X_va, y_va = va[FEATURES], va[TARGET].values
X_te, y_te = te[FEATURES], te[TARGET].values


# Logistic regression baseline
def for_lr(X):
    X = X.copy()
    X[CAT] = X[CAT].astype(str)
    return X


pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), NUM),
    ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=50), CAT),
])
logreg = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000, C=1.0))])
logreg.fit(for_lr(X_tr), y_tr)

# LightGBM
lgbm = lgb.LGBMClassifier(**LGBM_PARAMS)
lgbm.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], eval_metric="average_precision",
         callbacks=[lgb.early_stopping(100, verbose=False)])

# XGBoost
xgbm = xgb.XGBClassifier(n_estimators=2000, learning_rate=0.03, max_depth=6, subsample=0.8,
                         colsample_bytree=0.8, tree_method="hist", enable_categorical=True,
                         eval_metric="aucpr", early_stopping_rounds=100, random_state=42)
xgbm.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)

# Predictions and metrics
preds = {
    "logreg": (logreg.predict_proba(for_lr(X_va))[:, 1], logreg.predict_proba(for_lr(X_te))[:, 1]),
    "lgbm": (lgbm.predict_proba(X_va)[:, 1], lgbm.predict_proba(X_te)[:, 1]),
    "xgbm": (xgbm.predict_proba(X_va)[:, 1], xgbm.predict_proba(X_te)[:, 1]),
}
rows = []
for name, (p_va, p_te) in preds.items():
    rows.append({"model": name,
                 "val_pr_auc": average_precision_score(y_va, p_va),
                 "val_roc_auc": roc_auc_score(y_va, p_va),
                 "test_pr_auc": average_precision_score(y_te, p_te),
                 "test_roc_auc": roc_auc_score(y_te, p_te)})
res = pd.DataFrame(rows).round(4)
res["baseline_pr_auc_(prevalence)"] = round(y_te.mean(), 4)
print(res.to_string(index=False))
res.to_csv(PROC / "model_comparison.csv", index=False)

best = res.sort_values("val_pr_auc", ascending=False).iloc[0]["model"]
print("best model on validation PR-AUC:", best)

# Cost-sensitive threshold (assumptions in BRL, illustrative)
LATE_LOSS = 50.0    # expected loss per late order left unmanaged
SAVE_RATE = 0.30    # share of flagged late orders an intervention prevents
ACTION_COST = 3.0   # cost of one proactive intervention


def net_value(y, p, t):
    flag = p >= t
    return (y[flag] * SAVE_RATE * LATE_LOSS).sum() - ACTION_COST * flag.sum()


p_va, p_te = preds[best]
grid = np.linspace(0.02, 0.90, 89)
vals = [net_value(y_va, p_va, t) for t in grid]
t_star = float(grid[int(np.argmax(vals))])

flag_te = p_te >= t_star
tp = int((flag_te & (y_te == 1)).sum())
out = {
    "best_model": best,
    "threshold": round(t_star, 3),
    "test_flagged_share": round(float(flag_te.mean()), 4),
    "test_precision": round(tp / max(flag_te.sum(), 1), 4),
    "test_recall": round(tp / max(y_te.sum(), 1), 4),
    "test_net_value_model": round(float(net_value(y_te, p_te, t_star)), 1),
    "test_net_value_default_0.5": round(float(net_value(y_te, p_te, 0.5)), 1),
    "test_net_value_flag_everyone": round(float(net_value(y_te, p_te, 0.0)), 1),
}
print(json.dumps(out, indent=2))
json.dump(out, open(PROC / "threshold_result.json", "w"), indent=2)

# Save artefacts
joblib.dump(logreg, MODELS / "logreg.joblib")
joblib.dump(lgbm, MODELS / "lightgbm.joblib")
joblib.dump(xgbm, MODELS / "xgboost.joblib")

cols = ["order_id", "purchase_ts", "customer_state", "seller_state", "primary_seller_id",
        "primary_category_en", "total_price", "promised_days", "distance_km", "is_late", "review_score"]
pred_df = te[cols].copy()
pred_df["p_late"] = p_te
pred_df["flagged"] = flag_te.astype(int)
pred_df.to_csv(PROC / "test_predictions.csv", index=False)

train 67,519 | val 14,468 | test 14,469
late rate train/val/test: [np.float64(0.0783), np.float64(0.0431), np.float64(0.0429)]
date ranges: [(datetime.date(2016, 9, 15), datetime.date(2018, 4, 15)), (datetime.date(2018, 4, 15), datetime.date(2018, 6, 21)), (datetime.date(2018, 6, 21), datetime.date(2018, 8, 29))]
 model  val_pr_auc  val_roc_auc  test_pr_auc  test_roc_auc  baseline_pr_auc_(prevalence)
logreg      0.1249       0.7591       0.0739        0.6332                        0.0429
  lgbm      0.1324       0.7537       0.0728        0.6732                        0.0429
  xgbm      0.1297       0.7492       0.0740        0.6729                        0.0429
best model on validation PR-AUC: lgbm
{
  "best_model": "lgbm",
  "threshold": 0.15,
  "test_flagged_share": 0.0318,
  "test_precision": 0.087,
  "test_recall": 0.0645,
  "test_net_value_model": -780.0,
  "test_net_value_default_0.5": 0.0,
  "test_net_value_flag_everyone": -34107.0
}


In [ ]:
def net_value_p(y, p, t, loss, save, cost):
    flag = p >= t
    return (y[flag] * save * loss).sum() - cost * flag.sum()


grid_s = np.append(grid, 1.01)   # 1.01 means "flag nobody" (net value 0), so doing nothing is an option
rows = []
for loss in (25, 50, 100):
    for cost in (1, 3, 6):
        vals_s = [net_value_p(y_va, p_va, t, loss, SAVE_RATE, cost) for t in grid_s]
        t_s = float(grid_s[int(np.argmax(vals_s))])
        flagged = p_te >= t_s
        rows.append({"late_loss": loss, "action_cost": cost, "threshold": round(t_s, 3),
                     "flagged_share": round(float(flagged.mean()), 4),
                     "test_net_value": round(float(net_value_p(y_te, p_te, t_s, loss, SAVE_RATE, cost)), 1)})
sens = pd.DataFrame(rows)
print(sens.pivot(index="late_loss", columns="action_cost", values="test_net_value"))
sens.to_csv(PROC / "sensitivity.csv", index=False)

action_cost       1       3       6
late_loss                          
25           -687.5     0.0     0.0
50            301.0  -780.0     0.0
100          6870.0 -1605.0 -1560.0


## 6. Ablation study

In [ ]:
df = load_model_table()
tr, va, te = time_split(df)

sets = {
    "A. order features only": BASE_NUM + BASE_CAT,
    "B. A + geography": BASE_NUM + BASE_CAT + GEO_NUM + GEO_CAT,
    "C. B + seller history (full)": FEATURES,
}

rows = []
for name, feats in sets.items():
    m = lgb.LGBMClassifier(**LGBM_PARAMS)
    m.fit(tr[feats], tr[TARGET], eval_set=[(va[feats], va[TARGET])],
          eval_metric="average_precision", callbacks=[lgb.early_stopping(100, verbose=False)])
    p = m.predict_proba(te[feats])[:, 1]
    rows.append({"feature_set": name, "n_features": len(feats),
                 "test_pr_auc": average_precision_score(te[TARGET], p),
                 "test_roc_auc": roc_auc_score(te[TARGET], p)})

abl = pd.DataFrame(rows)
abl["pr_auc_gain_vs_A"] = (abl.test_pr_auc - abl.test_pr_auc.iloc[0]).round(4)
print(abl.round(4).to_string(index=False))
abl.to_csv(PROC / "ablation.csv", index=False)

                 feature_set  n_features  test_pr_auc  test_roc_auc  pr_auc_gain_vs_A
      A. order features only          17       0.0466        0.5156            0.0000
            B. A + geography          21       0.0481        0.4993            0.0015
C. B + seller history (full)          23       0.0728        0.6732            0.0262


## 7. SHAP, calibration, business impact

In [ ]:
import shap
from sklearn.calibration import calibration_curve

df = load_model_table()
tr, va, te = time_split(df)
lgbm = joblib.load(MODELS / "lightgbm.joblib")

# SHAP
Xs = te[FEATURES].sample(min(5000, len(te)), random_state=42)
sv = shap.TreeExplainer(lgbm).shap_values(Xs)
if isinstance(sv, list):
    sv = sv[1]
if getattr(sv, "ndim", 2) == 3:
    sv = sv[:, :, 1]
shap.summary_plot(sv, Xs, show=False, max_display=15)
plt.tight_layout(); plt.savefig(FIG / "shap_summary.png", dpi=200, bbox_inches="tight"); plt.close()   # >>> CHANGED: FIG

shap.summary_plot(sv, Xs, plot_type="bar", show=False, max_display=15)
plt.tight_layout(); plt.savefig(FIG / "shap_summary_bar.png", dpi=200, bbox_inches="tight"); plt.close()

# Calibration
p = lgbm.predict_proba(te[FEATURES])[:, 1]
frac_pos, mean_pred = calibration_curve(te[TARGET].values, p, n_bins=10, strategy="quantile")
plt.plot(mean_pred, frac_pos, marker="o", label="LightGBM")
plt.plot([0, max(mean_pred)], [0, max(mean_pred)], "--", color="grey", label="perfect")
plt.xlabel("Predicted probability"); plt.ylabel("Observed late rate"); plt.legend()
plt.title("Calibration (test set)")
plt.tight_layout(); plt.savefig(FIG / "calibration.png", dpi=200); plt.close()

# Business impact (descriptive, NOT causal)
rev = df.dropna(subset=["review_score"])
print("\nAverage review score by late flag:")
print(rev.groupby("is_late").review_score.agg(["mean", "count"]).round(3))

low = rev.assign(low_score=rev.review_score <= 2).groupby("is_late").low_score.mean().round(3)
print("\nShare of 1-2 star reviews:\n", low)

orders_per_cust = df.groupby("customer_unique_id").order_id.count()
first = df.sort_values("purchase_ts").drop_duplicates("customer_unique_id").copy()
first["repeat_buyer"] = first.customer_unique_id.map(orders_per_cust) > 1
print("\nRepeat purchase rate after a first order that was...")
print(first.groupby("is_late").repeat_buyer.mean().round(4))

# Revenue exposed to late delivery in the test period
pred = pd.read_csv(PROC / "test_predictions.csv")
print("\nTest period GMV:", round(pred.total_price.sum()))
print("GMV of orders that were actually late:", round(pred.loc[pred.is_late == 1, "total_price"].sum()))
print("GMV of flagged orders:", round(pred.loc[pred.flagged == 1, "total_price"].sum()))

/usr/local/lib/python3.13/dist-packages/shap/explainers/_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(



Average review score by late flag:
          mean  count
is_late              
0        4.290  89429
1        2.271   6381

Share of 1-2 star reviews:
 is_late
0    0.093
1    0.624
Name: low_score, dtype: float64

Repeat purchase rate after a first order that was...
is_late
0    0.0303
1    0.0253
Name: repeat_buyer, dtype: float64

Test period GMV: 1957134
GMV of orders that were actually late: 91843
GMV of flagged orders: 85700


## 8. Export for Tableau

In [ ]:
df = load_model_table()
for c in df.select_dtypes("category"):
    df[c] = df[c].astype(str)

order_cols = ["order_id", "purchase_ts", "customer_state", "seller_state", "primary_seller_id",
              "primary_category_en", "total_price", "total_freight", "promised_days",
              "distance_km", "delay_days", "is_late", "review_score"]
orders = df[order_cols].copy()
orders["purchase_month"] = orders["purchase_ts"].dt.to_period("M").dt.to_timestamp()
orders.to_csv(TAB / "bi_orders.csv", index=False)

pd.read_csv(PROC / "test_predictions.csv").to_csv(TAB / "bi_predictions.csv", index=False)

t = json.load(open(PROC / "threshold_result.json"))
print(f"bi_orders.csv: {len(orders):,} rows | bi_predictions.csv saved")
print("Tableau 'Threshold' parameter default:", t["threshold"])
print("Files are in:", TAB)

bi_orders.csv: 96,456 rows | bi_predictions.csv saved
Tableau 'Threshold' parameter default: 0.15
Files are in: /content/drive/MyDrive/olist-delivery-delay/data/processed/tableau
